# Аудит покрытия `dfs_5min.pickle`

Этот ноутбук углубляет первичный аудит: анализирует покрытие по колонкам `Date` и `Symbol`, регулярность 5-минутных баров, состав контрактов, дубликаты, нулевой объём и базовую корректность `OHLC (цены открытия, максимума, минимума и закрытия)`.

Исходный файл не изменяется. Результаты сохраняются в папку `coverage_audit_dfs_5min` рядом с ноутбуком.

In [1]:
from pathlib import Path
import json
import pickle
from collections import Counter

import numpy as np
import pandas as pd

In [2]:
DATA_PATH = Path('dfs_5min.pickle')
AUDIT_DIR = Path('coverage_audit_dfs_5min')
AUDIT_DIR.mkdir(exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Файл не найден рядом с ноутбуком: {DATA_PATH.resolve()}')

stat = DATA_PATH.stat()
pd.DataFrame([{
    'source_file': str(DATA_PATH.resolve()),
    'size_bytes': stat.st_size,
    'size_gb': stat.st_size / 1024**3,
    'modified_utc': pd.to_datetime(stat.st_mtime, unit='s', utc=True),
    'audit_dir': str(AUDIT_DIR.resolve()),
}])

,source_file,size_bytes,size_gb,modified_utc,audit_dir
0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,1458525408,1.358358,2026-09-12 09:26:11.031317234+00:00,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...


## Загрузка данных

Файл большой. Эта ячейка может занять время и несколько гигабайт оперативной памяти.

In [3]:
with DATA_PATH.open('rb') as f:
    dfs = pickle.load(f)

if not isinstance(dfs, dict):
    raise TypeError(f'Ожидался dict, получен {type(dfs).__name__}')

len(dfs), list(dfs.keys())[:10]

(35, ['ZC', 'ZS', 'ZL', 'ZM', 'ZW', 'KE', 'ZO', 'ZR', 'RS', 'CL'])

## Функции аудита

In [4]:
REQUIRED_COLUMNS = ['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume']


def safe_timedelta(value):
    return str(value) if pd.notna(value) else None


def audit_frame(name, frame):
    missing = [col for col in REQUIRED_COLUMNS if col not in frame.columns]
    if missing:
        return {'name': name, 'error': f'missing columns: {missing}'}

    date = pd.to_datetime(frame['Date'], errors='coerce')
    symbol = frame['Symbol'].astype('object')
    valid_date = date.notna()
    sorted_date = date[valid_date].sort_values()
    diffs = sorted_date.diff().dropna()
    duplicate_date_symbol = frame.assign(_Date=date).duplicated(['_Date', 'Symbol']).sum()

    ohlc = frame[['Open', 'High', 'Low', 'Close']].apply(pd.to_numeric, errors='coerce')
    volume = pd.to_numeric(frame['Volume'], errors='coerce')
    bad_ohlc = (
        (ohlc['High'] < ohlc['Low'])
        | (ohlc['Open'] > ohlc['High']) | (ohlc['Open'] < ohlc['Low'])
        | (ohlc['Close'] > ohlc['High']) | (ohlc['Close'] < ohlc['Low'])
    )

    return {
        'name': name,
        'rows': int(len(frame)),
        'columns': list(frame.columns),
        'date_na': int(date.isna().sum()),
        'start': date.min(),
        'end': date.max(),
        'calendar_days': int((date.max() - date.min()).days) if date.notna().any() else None,
        'unique_dates': int(date.nunique(dropna=True)),
        'unique_symbols': int(symbol.nunique(dropna=True)),
        'duplicate_date_symbol': int(duplicate_date_symbol),
        'median_step': diffs.median() if len(diffs) else pd.NaT,
        'mode_step': diffs.mode().iloc[0] if len(diffs) and not diffs.mode().empty else pd.NaT,
        'share_5min_steps_global': float((diffs == pd.Timedelta(minutes=5)).mean()) if len(diffs) else np.nan,
        'largest_gap_global': diffs.max() if len(diffs) else pd.NaT,
        'zero_volume_rows': int((volume == 0).sum()),
        'zero_volume_pct': float((volume == 0).mean()),
        'volume_na': int(volume.isna().sum()),
        'bad_ohlc_rows': int(bad_ohlc.sum()),
        'bad_ohlc_pct': float(bad_ohlc.mean()),
    }


def audit_symbols(name, frame):
    date = pd.to_datetime(frame['Date'], errors='coerce')
    volume = pd.to_numeric(frame['Volume'], errors='coerce')
    work = pd.DataFrame({'Date': date, 'Symbol': frame['Symbol'], 'Volume': volume})
    rows = []
    for sym, part in work.groupby('Symbol', dropna=False):
        idx = part['Date'].dropna().sort_values()
        diffs = idx.diff().dropna()
        rows.append({
            'name': name,
            'symbol': sym,
            'rows': int(len(part)),
            'start': idx.min() if len(idx) else pd.NaT,
            'end': idx.max() if len(idx) else pd.NaT,
            'unique_dates': int(idx.nunique()),
            'median_step': diffs.median() if len(diffs) else pd.NaT,
            'share_5min_steps': float((diffs == pd.Timedelta(minutes=5)).mean()) if len(diffs) else np.nan,
            'largest_gap': diffs.max() if len(diffs) else pd.NaT,
            'zero_volume_pct': float((part['Volume'] == 0).mean()),
            'volume_sum': float(part['Volume'].sum(skipna=True)),
        })
    return rows

## Сводка по рынкам

In [5]:
market_rows = []
symbol_rows = []

for name, frame in dfs.items():
    if not isinstance(frame, pd.DataFrame):
        market_rows.append({'name': name, 'error': f'value is {type(frame).__name__}'})
        continue
    market_rows.append(audit_frame(name, frame))
    if all(col in frame.columns for col in ['Date', 'Symbol', 'Volume']):
        symbol_rows.extend(audit_symbols(name, frame))

market_coverage = pd.DataFrame(market_rows)
symbol_coverage = pd.DataFrame(symbol_rows)

market_coverage.sort_values('rows', ascending=False).head(40)

,name,rows,columns,date_na,start,end,calendar_days,unique_dates,unique_symbols,duplicate_date_symbol,median_step,mode_step,share_5min_steps_global,largest_gap_global,zero_volume_rows,zero_volume_pct,volume_na,bad_ohlc_rows,bad_ohlc_pct
30,J6,1048570,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-09-29 19:20:00,2023-08-10 11:00:00,5427,1048570,60,0,0 days 00:05:00,0 days 00:05:00,0.994980,3 days 13:05:00,0,0.000000,0,0,0.0
29,A6,1048570,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-07-01 07:25:00,2023-08-10 11:00:00,5518,1048570,61,0,0 days 00:05:00,0 days 00:05:00,0.992445,3 days 13:05:00,0,0.000000,0,0,0.0
9,CL,1048570,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-11-06 04:45:00,2023-08-10 11:00:00,5390,1048570,178,0,0 days 00:05:00,0 days 00:05:00,0.995171,3 days 04:20:00,0,0.000000,0,0,0.0
27,B6,1048570,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-07-14 03:05:00,2023-08-10 11:00:00,5505,1048570,61,0,0 days 00:05:00,0 days 00:05:00,0.990360,3 days 13:05:00,0,0.000000,0,0,0.0
25,E6,1048570,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-10-07 03:00:00,2023-08-10 11:00:00,5420,1048570,60,0,0 days 00:05:00,0 days 00:05:00,0.995665,3 days 23:05:00,0,0.000000,0,0,0.0
26,D6,1043303,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-05-04 20:20:00,2023-08-10 11:00:00,5575,1043303,62,0,0 days 00:05:00,0 days 00:05:00,0.986700,3 days 13:05:00,0,0.000000,0,0,0.0
31,S6,1020486,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-05-04 20:20:00,2023-08-10 11:00:00,5575,1020486,62,0,0 days 00:05:00,0 days 00:05:00,0.967864,3 days 13:05:00,0,0.000000,0,0,0.0
12,NG,959491,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-05-04 20:20:00,2023-08-10 11:00:00,5575,959491,177,0,0 days 00:05:00,0 days 00:05:00,0.942830,3 days 17:35:00,0,0.000000,0,0,0.0
32,N6,957654,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-05-05 04:55:00,2023-08-10 11:00:00,5575,957654,62,0,0 days 00:05:00,0 days 00:05:00,0.954451,13 days 22:35:00,0,0.000000,0,0,0.0
28,DX,935991,"[Date, Symbol, Open, High, Low, Close, Volume,...",0,2008-05-04 20:20:00,2023-08-10 11:00:00,5575,935991,62,0,0 days 00:05:00,0 days 00:05:00,0.966930,3 days 15:55:00,0,0.000000,0,0,0.0


In [6]:
symbol_coverage.sort_values(['name', 'start', 'symbol']).head(80)

,name,symbol,rows,start,end,unique_dates,median_step,share_5min_steps,largest_gap,zero_volume_pct,volume_sum
2921,A6,A6U08,3503,2008-07-01 07:25:00,2008-09-11 15:45:00,3503,0 days 00:05:00,0.559395,2 days 14:10:00,0.0,9094.0
2937,A6,A6Z08,9332,2008-09-12 01:30:00,2008-12-11 23:55:00,9332,0 days 00:05:00,0.887793,2 days 12:35:00,0.0,708808.0
2891,A6,A6H09,16134,2008-12-12 00:00:00,2009-03-11 23:50:00,16134,0 days 00:05:00,0.963801,2 days 01:50:00,0.0,1700424.0
2906,A6,A6M09,17334,2009-03-12 00:10:00,2009-06-10 23:55:00,17334,0 days 00:05:00,0.984192,3 days 01:50:00,0.0,3047724.0
2922,A6,A6U09,17600,2009-06-11 00:00:00,2009-09-09 23:50:00,17600,0 days 00:05:00,0.989659,2 days 05:05:00,0.0,4543029.0
...,...,...,...,...,...,...,...,...,...,...,...
2771,B6,B6H12,17140,2011-12-14 00:05:00,2012-03-14 23:55:00,17140,0 days 00:05:00,0.992998,3 days 13:05:00,0.0,4580464.0
2786,B6,B6M12,18025,2012-03-15 00:05:00,2012-06-14 23:55:00,18025,0 days 00:05:00,0.993675,2 days 06:50:00,0.0,6054912.0
2802,B6,B6U12,17738,2012-06-15 00:00:00,2012-09-13 23:55:00,17738,0 days 00:05:00,0.991318,2 days 01:50:00,0.0,5999060.0
2818,B6,B6Z12,17508,2012-09-14 00:00:00,2012-12-12 23:55:00,17508,0 days 00:05:00,0.993317,2 days 04:50:00,0.0,5801560.0


## Структура контрактов внутри каждого рынка

In [7]:
contract_count_summary = (
    symbol_coverage
    .groupby('name', as_index=False)
    .agg(
        contracts=('symbol', 'nunique'),
        first_date=('start', 'min'),
        last_date=('end', 'max'),
        total_rows=('rows', 'sum'),
        median_rows_per_contract=('rows', 'median'),
        min_rows_per_contract=('rows', 'min'),
        max_rows_per_contract=('rows', 'max'),
    )
    .sort_values(['contracts', 'total_rows'], ascending=False)
)
contract_count_summary

,name,contracts,first_date,last_date,total_rows,median_rows_per_contract,min_rows_per_contract,max_rows_per_contract
12,HO,184,2008-05-05 08:00:00,2023-08-10 11:00:00,886164,4938.0,254,6366
22,RB,184,2008-05-04 20:20:00,2023-08-10 11:00:00,840805,4735.5,131,6438
17,LF,183,2008-05-05 08:20:00,2023-08-10 11:00:00,854992,4771.0,291,9180
4,CL,178,2008-11-06 04:45:00,2023-08-10 11:00:00,1048570,5915.0,2172,8700
20,NG,177,2008-05-04 20:20:00,2023-08-10 11:00:00,959491,5329.0,820,11260
6,DL,163,2008-11-03 09:10:00,2023-08-10 11:00:00,146705,890.0,25,3787
15,L6,145,2008-11-02 18:40:00,2023-08-10 11:00:00,218100,1550.0,3,3439
21,QA,132,2009-09-08 08:00:00,2023-08-10 11:00:00,519049,4100.5,2,17405
10,GF,117,2008-05-05 09:05:00,2023-08-10 11:00:00,275826,1829.0,90,10013
11,HE,109,2008-05-05 09:05:00,2023-08-10 11:00:00,348686,2420.0,554,7187


## Кандидаты для первой волны стратегий

Это не торговая рекомендация. Это технический список рынков, где покрытие выглядит достаточным для первичного исследования.

In [8]:
candidate_markets = market_coverage.copy()
candidate_markets['usable_for_first_pass'] = (
    (candidate_markets['rows'] >= 250_000)
    & (candidate_markets['unique_symbols'] >= 3)
    & (candidate_markets['date_na'] == 0)
    & (candidate_markets['bad_ohlc_rows'] == 0)
)

candidate_markets[[
    'name', 'usable_for_first_pass', 'rows', 'unique_symbols', 'start', 'end',
    'duplicate_date_symbol', 'zero_volume_pct', 'bad_ohlc_rows', 'share_5min_steps_global'
]].sort_values(['usable_for_first_pass', 'rows'], ascending=False)

,name,usable_for_first_pass,rows,unique_symbols,start,end,duplicate_date_symbol,zero_volume_pct,bad_ohlc_rows,share_5min_steps_global
9,CL,True,1048570,178,2008-11-06 04:45:00,2023-08-10 11:00:00,0,0.000000,0,0.995171
25,E6,True,1048570,60,2008-10-07 03:00:00,2023-08-10 11:00:00,0,0.000000,0,0.995665
27,B6,True,1048570,61,2008-07-14 03:05:00,2023-08-10 11:00:00,0,0.000000,0,0.990360
29,A6,True,1048570,61,2008-07-01 07:25:00,2023-08-10 11:00:00,0,0.000000,0,0.992445
30,J6,True,1048570,60,2008-09-29 19:20:00,2023-08-10 11:00:00,0,0.000000,0,0.994980
26,D6,True,1043303,62,2008-05-04 20:20:00,2023-08-10 11:00:00,0,0.000000,0,0.986700
31,S6,True,1020486,62,2008-05-04 20:20:00,2023-08-10 11:00:00,0,0.000000,0,0.967864
12,NG,True,959491,177,2008-05-04 20:20:00,2023-08-10 11:00:00,0,0.000000,0,0.942830
32,N6,True,957654,62,2008-05-05 04:55:00,2023-08-10 11:00:00,0,0.000000,0,0.954451
28,DX,True,935991,62,2008-05-04 20:20:00,2023-08-10 11:00:00,0,0.000000,0,0.966930


## Сохранение результатов

In [9]:
market_coverage.to_csv(AUDIT_DIR / '01_market_coverage.csv', index=False, encoding='utf-8-sig')
symbol_coverage.to_csv(AUDIT_DIR / '02_symbol_coverage.csv', index=False, encoding='utf-8-sig')
contract_count_summary.to_csv(AUDIT_DIR / '03_contract_count_summary.csv', index=False, encoding='utf-8-sig')
candidate_markets.to_csv(AUDIT_DIR / '04_candidate_markets.csv', index=False, encoding='utf-8-sig')

manifest = {
    'source_file': str(DATA_PATH.resolve()),
    'source_size_bytes': int(stat.st_size),
    'audit_dir': str(AUDIT_DIR.resolve()),
    'markets': int(len(dfs)),
    'market_rows': int(len(market_coverage)),
    'symbol_rows': int(len(symbol_coverage)),
    'saved_files': sorted(path.name for path in AUDIT_DIR.glob('*.csv')),
}

with (AUDIT_DIR / '00_manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=str)

readme = '''# Coverage audit for dfs_5min.pickle

Generated by `audit_dfs_5min_coverage.ipynb`.

Files:

- `00_manifest.json` — compact machine-readable summary.
- `01_market_coverage.csv` — one row per top-level market key.
- `02_symbol_coverage.csv` — one row per concrete `Symbol` inside each market.
- `03_contract_count_summary.csv` — contract count and row distribution by market.
- `04_candidate_markets.csv` — rough technical candidates for the first research pass.

The source pickle is not modified.
'''
(AUDIT_DIR / 'README.md').write_text(readme, encoding='utf-8')

pd.DataFrame({'saved_file': manifest['saved_files'] + ['00_manifest.json', 'README.md']})

,saved_file
0,01_market_coverage.csv
1,02_symbol_coverage.csv
2,03_contract_count_summary.csv
3,04_candidate_markets.csv
4,00_manifest.json
5,README.md
